# SageMaker Endpoint Deployment & Testing

Tento notebook ukazuje, ako:
1. Deploy model na SageMaker real-time endpoint
2. Test endpoint s real-time predictions
3. Monitor endpoint metrics
4. Update endpoint s novou verziou modelu
5. A/B testing s multiple model variants

## Prerekvizity

1. ✅ Trained model v S3 (z KROK 4)
2. ✅ Docker image v ECR
3. ✅ SageMaker execution role (Terraform output)
4. ✅ (Optional) Deployed endpoint Terraform infrastructure

## 1. Setup

In [ ]:
import os
import json
import boto3
import sagemaker
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
import pandas as pd
import numpy as np
import subprocess
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# AWS Session
boto_session = boto3.Session(region_name='eu-west-1')
sagemaker_session = sagemaker.Session(boto_session=boto_session)
sm_client = boto3.client('sagemaker', region_name='eu-west-1')
sm_runtime = boto3.client('sagemaker-runtime', region_name='eu-west-1')
cw_client = boto3.client('cloudwatch', region_name='eu-west-1')

region = boto_session.region_name
print(f"AWS Region: {region}")

## 2. Load Configuration

Načítame konfiguráciu z Terraform outputs.

In [ ]:
def get_terraform_output(key: str, tf_dir: str = "../infra/terraform/sagemaker") -> str:
    """Získaj hodnotu z Terraform output."""
    result = subprocess.run(
        ["terraform", "output", "-raw", key],
        cwd=tf_dir,
        capture_output=True,
        text=True
    )
    if result.returncode != 0:
        raise RuntimeError(f"Failed to get Terraform output '{key}': {result.stderr}")
    return result.stdout.strip()

# Get Terraform outputs from training infrastructure
role_arn = get_terraform_output("sagemaker_execution_role_arn")
ecr_repository_url = get_terraform_output("ecr_repository_url")
s3_model_bucket = get_terraform_output("s3_model_bucket_name")

print(f"SageMaker Role ARN: {role_arn}")
print(f"ECR Repository: {ecr_repository_url}")
print(f"S3 Model Bucket: {s3_model_bucket}")

## 3. Specify Model Artifacts

Nastav S3 URI pre model artifacts (output z training jobu).

In [ ]:
# Option 1: Manually set model S3 URI (from training job output)
# model_data_url = "s3://your-bucket/training-jobs/job-name/output/model.tar.gz"

# Option 2: Get latest training job output
def get_latest_training_job_model(bucket_name: str, prefix: str = "training-jobs") -> str:
    """Nájde najnovší model z training jobs v S3."""
    s3_client = boto3.client('s3')
    
    # List all training job outputs
    response = s3_client.list_objects_v2(
        Bucket=bucket_name,
        Prefix=prefix,
        Delimiter='/'
    )
    
    if 'CommonPrefixes' not in response:
        raise ValueError(f"No training jobs found in s3://{bucket_name}/{prefix}")
    
    # Get latest job (sorted by name)
    job_prefixes = [p['Prefix'] for p in response['CommonPrefixes']]
    latest_job = sorted(job_prefixes)[-1]
    
    # Model is typically at: {prefix}/{job-name}/output/model.tar.gz
    model_key = f"{latest_job}output/model.tar.gz"
    
    # Verify model exists
    try:
        s3_client.head_object(Bucket=bucket_name, Key=model_key)
        return f"s3://{bucket_name}/{model_key}"
    except:
        raise FileNotFoundError(f"Model not found at s3://{bucket_name}/{model_key}")

# Try to find latest model
try:
    model_data_url = get_latest_training_job_model(s3_model_bucket)
    print(f"✓ Found latest model: {model_data_url}")
except Exception as e:
    print(f"⚠️  Could not auto-detect model: {e}")
    print("Please set model_data_url manually")
    # model_data_url = "s3://..."

## 4. Create SageMaker Model

In [ ]:
# Model configuration
model_name = f"house-price-model-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
ecr_image_uri = f"{ecr_repository_url}:latest"

print(f"Model Name: {model_name}")
print(f"Model Data: {model_data_url}")
print(f"ECR Image: {ecr_image_uri}")

# Create Model
model = Model(
    model_data=model_data_url,
    image_uri=ecr_image_uri,
    role=role_arn,
    name=model_name,
    env={
        "SAGEMAKER_PROGRAM": "src/serve/inference.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/code",
        "MODEL_SERVER_TIMEOUT": "120",
        "MODEL_SERVER_WORKERS": "1",
    },
    sagemaker_session=sagemaker_session
)

print("✓ Model object created")

## 5. Deploy Endpoint

Deploy model na real-time endpoint.

**Note:** Deployment trvá ~5-7 minút.

In [ ]:
# Endpoint configuration
endpoint_name = "house-price-mlops-prod-endpoint"
instance_type = "ml.t2.medium"  # ~$0.065/hour
instance_count = 1

print(f"Deploying endpoint: {endpoint_name}")
print(f"Instance Type: {instance_type}")
print(f"Instance Count: {instance_count}")
print("\nThis may take 5-7 minutes...")

# Deploy (wait=True blocks until InService)
predictor = model.deploy(
    initial_instance_count=instance_count,
    instance_type=instance_type,
    endpoint_name=endpoint_name,
    serializer=JSONSerializer(),
    deserializer=JSONDeserializer(),
    wait=True,  # Wait for endpoint to be InService
)

print("\n✓ Endpoint deployed successfully!")
print(f"Endpoint Name: {predictor.endpoint_name}")

## 6. Test Endpoint - Single Prediction

In [ ]:
# Load test data
test_data = pd.read_csv("../data/test.csv")
print(f"Test data shape: {test_data.shape}")

# Select first sample
sample = test_data.iloc[0].to_dict()
print(f"\nSample features: {list(sample.keys())[:10]}...")

# Prepare payload
payload = {"data": [list(sample.values())]}

# Invoke endpoint
response = predictor.predict(payload)

# Parse prediction (model predicts log(SalePrice))
log_prediction = response['predictions'][0]
prediction = np.exp(log_prediction)

print(f"\n=== Prediction ===")
print(f"Log(SalePrice): {log_prediction:.4f}")
print(f"Predicted SalePrice: ${prediction:,.2f}")

## 7. Batch Predictions

Test endpoint s batch requestami.

In [ ]:
# Batch prediction (first 10 samples)
batch_size = 10
batch_samples = test_data.iloc[:batch_size]

# Convert to list of lists
batch_data = batch_samples.values.tolist()
batch_payload = {"data": batch_data}

# Invoke endpoint
print(f"Sending batch of {batch_size} samples...")
batch_response = predictor.predict(batch_payload)

# Parse predictions
log_predictions = batch_response['predictions']
predictions = np.exp(log_predictions)

# Create results DataFrame
results = pd.DataFrame({
    'ID': test_data.iloc[:batch_size]['Id'],
    'Log_Prediction': log_predictions,
    'Predicted_Price': predictions
})

print("\n=== Batch Predictions ===")
print(results)
print(f"\nAverage Predicted Price: ${predictions.mean():,.2f}")
print(f"Min: ${predictions.min():,.2f}, Max: ${predictions.max():,.2f}")

## 8. Measure Latency

Test endpoint latency.

In [ ]:
import time

# Prepare single sample
sample_payload = {"data": [test_data.iloc[0].values.tolist()]}

# Warm-up request
predictor.predict(sample_payload)

# Measure latency (10 requests)
latencies = []
num_requests = 10

print(f"Measuring latency over {num_requests} requests...")
for i in range(num_requests):
    start = time.time()
    predictor.predict(sample_payload)
    latency = (time.time() - start) * 1000  # Convert to ms
    latencies.append(latency)
    
latencies = np.array(latencies)

print("\n=== Latency Statistics ===")
print(f"Mean: {latencies.mean():.2f} ms")
print(f"Median: {np.median(latencies):.2f} ms")
print(f"p95: {np.percentile(latencies, 95):.2f} ms")
print(f"p99: {np.percentile(latencies, 99):.2f} ms")
print(f"Min: {latencies.min():.2f} ms")
print(f"Max: {latencies.max():.2f} ms")

# Plot latency distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(latencies, bins=20, edgecolor='black')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Latency Distribution')

plt.subplot(1, 2, 2)
plt.plot(latencies, marker='o')
plt.xlabel('Request Number')
plt.ylabel('Latency (ms)')
plt.title('Latency Over Time')
plt.tight_layout()
plt.show()

## 9. Monitor Endpoint Metrics

Check CloudWatch metrics pre endpoint.

In [ ]:
def get_endpoint_metrics(endpoint_name: str, metric_name: str, hours: int = 1):
    """Získaj CloudWatch metriky pre endpoint."""
    end_time = datetime.utcnow()
    start_time = end_time - timedelta(hours=hours)
    
    response = cw_client.get_metric_statistics(
        Namespace='AWS/SageMaker',
        MetricName=metric_name,
        Dimensions=[
            {'Name': 'EndpointName', 'Value': endpoint_name},
            {'Name': 'VariantName', 'Value': 'AllTraffic'}
        ],
        StartTime=start_time,
        EndTime=end_time,
        Period=300,  # 5 minutes
        Statistics=['Sum', 'Average']
    )
    
    return response['Datapoints']

# Get metrics
invocations = get_endpoint_metrics(endpoint_name, 'Invocations')
model_latency = get_endpoint_metrics(endpoint_name, 'ModelLatency')
errors_4xx = get_endpoint_metrics(endpoint_name, 'Invocation4XXErrors')
errors_5xx = get_endpoint_metrics(endpoint_name, 'Invocation5XXErrors')

print(f"=== Endpoint Metrics (last hour) ===")
print(f"Invocations: {sum([d['Sum'] for d in invocations]):.0f}")
print(f"Avg Latency: {np.mean([d['Average'] for d in model_latency]) if model_latency else 0:.2f} ms")
print(f"4XX Errors: {sum([d['Sum'] for d in errors_4xx]):.0f}")
print(f"5XX Errors: {sum([d['Sum'] for d in errors_5xx]):.0f}")

## 10. Check Endpoint Status

In [ ]:
# Describe endpoint
endpoint_description = sm_client.describe_endpoint(EndpointName=endpoint_name)

print("=== Endpoint Status ===")
print(f"Name: {endpoint_description['EndpointName']}")
print(f"Status: {endpoint_description['EndpointStatus']}")
print(f"ARN: {endpoint_description['EndpointArn']}")
print(f"Created: {endpoint_description['CreationTime']}")
print(f"Last Modified: {endpoint_description['LastModifiedTime']}")

# Production variants
print("\n=== Production Variants ===")
for variant in endpoint_description['ProductionVariants']:
    print(f"\nVariant: {variant['VariantName']}")
    print(f"  Instance Type: {variant['InstanceType']}")
    print(f"  Desired Instances: {variant['DesiredInstanceCount']}")
    print(f"  Current Instances: {variant['CurrentInstanceCount']}")
    print(f"  Weight: {variant['CurrentWeight']}")

## 11. Update Endpoint (Optional)

Aktualizuj endpoint s novou verziou modelu (zero-downtime deployment).

In [ ]:
# UNCOMMENT to update endpoint with new model

# # New model configuration
# new_model_name = f"house-price-model-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
# new_model_data_url = "s3://your-bucket/new-model/model.tar.gz"  # Set this!

# # Create new model
# new_model = Model(
#     model_data=new_model_data_url,
#     image_uri=ecr_image_uri,
#     role=role_arn,
#     name=new_model_name,
#     env={
#         "SAGEMAKER_PROGRAM": "src/serve/inference.py",
#         "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/code",
#     },
#     sagemaker_session=sagemaker_session
# )

# # Update endpoint (rolling update with zero downtime)
# print(f"Updating endpoint {endpoint_name} with new model...")
# print("This performs a rolling update with zero downtime")

# predictor = new_model.deploy(
#     initial_instance_count=instance_count,
#     instance_type=instance_type,
#     endpoint_name=endpoint_name,
#     update_endpoint=True,  # Important!
#     wait=True,
# )

# print("✓ Endpoint updated successfully!")

## 12. Cleanup

Delete endpoint to stop incurring charges.

In [ ]:
# UNCOMMENT to delete endpoint

# # Delete endpoint
# print(f"Deleting endpoint: {endpoint_name}...")
# predictor.delete_endpoint()
# print("✓ Endpoint deleted")

# # Optionally delete model
# print(f"Deleting model: {model_name}...")
# predictor.delete_model()
# print("✓ Model deleted")

print("\nTo delete endpoint:")
print(f"  predictor.delete_endpoint()")
print("\nOr via AWS CLI:")
print(f"  aws sagemaker delete-endpoint --endpoint-name {endpoint_name}")

## Summary

V tomto notebooku si:

1. ✅ Deployol model na SageMaker real-time endpoint
2. ✅ Testoval endpoint s single a batch predictions
3. ✅ Zmeral latency a performance
4. ✅ Monitoroval CloudWatch metrics
5. ✅ Skontroloval endpoint status

## Next Steps

- **Auto-scaling**: Nastav auto-scaling policy (Terraform)
- **A/B Testing**: Deploy multiple variants s traffic splitting
- **Model Monitoring**: Enable data capture a SageMaker Model Monitor
- **Load Testing**: Test endpoint pod záťažou (Locust, JMeter)
- **CI/CD**: Automatizuj deployment cez GitHub Actions

## Cost

**ml.t2.medium endpoint:**
- Hourly: $0.065
- Daily: ~$1.56
- Monthly: ~$47

**Don't forget to delete endpoint when not in use!**